In [1]:
import spacy
from spacy.matcher import PhraseMatcher
import psycopg2
import pdfplumber

In [ ]:


# Lazy load spaCy
nlp = spacy.load("en_core_web_sm")

# Region matcher
regions = [
    "Amhara", "Afar", "Oromia", "Tigray", "Somali",
    "Benishangul-Gumuz", "Sidama", "Harari", "Gambela",
    "Addis Ababa", "Dire Dawa"
]
matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
patterns = [nlp.make_doc(region) for region in regions]
matcher.add("REGION", patterns)

# Source
source_map = {
    "pdf": " DHS Final Report (2016 E.C.)",
    "csv": " Socioeconomic Survey (2021–2022)"
}


In [5]:
# ✅ Load PDF once and pre-index keywords
pdf_path = r"C:\Users\Hamza\Downloads\intern\2016-ethiopian-dhs-final-report.pdf"
pdf_text = ""
with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        page_text = page.extract_text()
        if page_text:
            pdf_text += page_text.lower() + "\n"

pdf_index = {}
for keyword in ["fertility", "education", "health", "definition"]:
    pos = pdf_text.find(keyword)
    if pos != -1:
        pdf_index[keyword] = pdf_text[pos:pos+300]


In [6]:
# ✅ Query helper
def query_postgres(sql):
    with psycopg2.connect(
        dbname="House Hold Data",
        user="postgres",
        password="newpassword",
        host="localhost",
        port="5432"
    ) as conn:
        with conn.cursor() as cur:
            cur.execute(sql)
            return cur.fetchall()

In [7]:
# Cache dictionary
cache = {}

def ask_question(user_question: str):
    """Answer one question and stop automatically."""
    if not user_question.strip():
        return "👋 Chatbot stopped (no question entered)"

    # Cache check
    if user_question in cache:
        return cache[user_question]

    doc = nlp(user_question)
    matches = matcher(doc)
    detected_regions = [doc[start:end].text for match_id, start, end in matches]

    answer = None

    # Definitions from PDF
    if "definition" in user_question.lower() or "explain" in user_question.lower():
        keyword = [token.text for token in doc if token.pos_ in ["NOUN", "PROPN"]][-1]
        snippet = pdf_index.get(keyword.lower(), "No definition found.")
        answer = f"{source_map['pdf']}: {snippet}"

    # Married households by region
    elif "married" in user_question.lower() and detected_regions:
        region = detected_regions[0].upper()
        sql = f"SELECT COUNT(*) FROM merged_household_data WHERE region = '{region}' AND marital_status = 'MARRIED';"
        result = query_postgres(sql)
        answer = f"{source_map['csv']} — Married households in {region}: {result}"

    # Education stats
    elif "education" in user_question.lower() and detected_regions:
        region = detected_regions[0].upper()
        sql = f"SELECT AVG(years_of_schooling) FROM merged_household_data WHERE region = '{region}';"
        result = query_postgres(sql)
        answer = f"{source_map['csv']} — Education stats for {region}: {result}"

    # Health queries
    elif "health" in user_question.lower():
        sql = "SELECT COUNT(*) FROM merged_household_data WHERE health_condition IS NOT NULL;"
        result = query_postgres(sql)
        answer = f"{source_map['csv']} — Health records: {result}"

    # Fertility queries (PDF definitions)
    elif "fertility" in user_question.lower():
        snippet = pdf_index.get("fertility", "No fertility info found.")
        answer = f"{source_map['pdf']}: {snippet}"

    # Income queries
    elif "income" in user_question.lower() and detected_regions:
        region = detected_regions[0].upper()
        sql = f"SELECT AVG(income) FROM merged_household_data WHERE region = '{region}';"
        result = query_postgres(sql)
        answer = f"{source_map['csv']} — Average income in {region}: {result}"

    # Household size queries
    elif "household size" in user_question.lower() and detected_regions:
        region = detected_regions[0].upper()
        sql = f"SELECT AVG(household_size) FROM merged_household_data WHERE region = '{region}';"
        result = query_postgres(sql)
        answer = f"{source_map['csv']} — Average household size in {region}: {result}"

    else:
        answer = "Sorry, I don't know how to answer that yet."

    # ✅ Cache the answer
    cache[user_question] = answer
    return answer